# 90 · 병합 + 파생 컬럼

| | |
|---|---|
| **목적** | `data/processed/` 를 전부 로드해 **합칠 수 있는 것만** 합치고, 못 합치는 이유를 남긴다 |
| **input** | `data-analysis/data/processed/*.parquet` (6개) |
| **output** | `data-analysis/data/processed/kseis_all.parquet`<br>`data-analysis/data/processed/joonggomall_with_category.parquet`<br>`data-analysis/reports/merge_decisions.json` |

> 🚨 **결론을 먼저 적는다: 6개 데이터셋을 하나로 join 할 공통 키는 없다.**
> 억지로 만들면 없는 관계를 지어내는 것이다(D65 와 같은 태도). 이 노트북은
> **① 진짜 합쳐지는 것(조달청 2종 세로 결합)** 과 **② 합쳐지지 않는 것 + 이유** 를 나눠 기록한다.

In [ ]:
import sys
sys.path.insert(0, "..")
import json
import re

import pandas as pd
from common import PROCESSED, REPORTS, REPO_ROOT

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 1. 로드 — `data/processed/` 전량

In [ ]:
sets = {}
for p in sorted(PROCESSED.glob("*.parquet")):
    sets[p.stem] = pd.read_parquet(p)

rows = [{"데이터셋": k, "행": len(v), "열": len(v.columns)} for k, v in sets.items()]
print(pd.DataFrame(rows).to_string(index=False))
print(f"\n총 {len(sets)}개 · {sum(len(v) for v in sets.values()):,}행")

## 2. 조인 가능성 점검 — 공통 컬럼이 있는가

먼저 **기계적으로** 확인한다. 컬럼 이름이 겹치는 쌍이 있는지.

In [ ]:
names = list(sets)
print(f"{'A':26}{'B':26}공통 컬럼")
print("-" * 76)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        공통 = set(sets[a].columns) & set(sets[b].columns)
        if 공통:
            print(f"{a:26}{b:26}{sorted(공통)}")
print()
print("→ 조달청 2종을 빼면 겹치는 컬럼이 없다. 이름이 겹쳐도 의미가 같아야 조인이다.")

## 3. ✅ 합쳐지는 것 — 조달청 2종 **세로 결합(concat)**

`kseis_machinery` 와 `kseis_electrical` 은 **같은 파일 포맷의 분야만 다른 배포본**이다.
컬럼 12개가 완전히 동일하므로 **join 이 아니라 concat** 이 맞다.

| 항목 | 값 |
|---|---|
| 방식 | **세로 결합** (`pd.concat`) — join 아님 |
| 키 | 없음 (스키마 동일) |
| 중복 검사 | `물품식별번호` 로 확인 |

In [ ]:
m, e = sets["kseis_machinery"], sets["kseis_electrical"]
print("컬럼 동일:", list(m.columns) == list(e.columns))
print(f"결합 전: machinery {len(m):,}행 + electrical {len(e):,}행 = {len(m) + len(e):,}행")

kseis_all = pd.concat([m, e], ignore_index=True)
dup = kseis_all["물품식별번호"].duplicated().sum()
print(f"결합 후: {len(kseis_all):,}행  (물품식별번호 중복 {dup}건)")
assert len(kseis_all) == len(m) + len(e), "행이 새거나 늘었다"
print("\n분야별 —")
print(kseis_all["분야"].value_counts().to_string())

### 3-1. 파생 — `단가_원_per_단위`

단위(`m`/`개`/`대`)가 섞여 있어 **단위를 무시한 가격 비교는 무의미**하다.
단위를 명시적으로 들고 다니는 파생 컬럼을 만든다.

In [ ]:
import numpy as np

kseis_all["단가_표기"] = kseis_all["가격"].map("{:,}".format) + "원/" + kseis_all["단위"]
# 가격이 13원 ~ 55만원으로 4자릿수 넘게 퍼져 있어 로그축을 같이 둔다
kseis_all["가격_로그"] = np.log10(kseis_all["가격"].clip(lower=1)).round(3)

print(kseis_all[["품명", "규격", "단위", "가격", "단가_표기", "가격_로그"]].head(5).to_string(index=False))
print()
print("단위별 가격 중앙값 —")
print(kseis_all.groupby("단위")["가격"].agg(["size", "median"]).sort_values("size", ascending=False).to_string())

## 4. 🚨 합쳐지지 않는 것 — 중진공 ↔ 조달청

**설계 승인 사항이었던 "카테고리 매핑 테이블 수기 정의"를 실제로 시도하고, 결과를 그대로 남긴다.**

### 4-1. 먼저 — 중진공 `카테고리1` 자체에 표기 변종이 있다

In [ ]:
j = sets["joonggomall_assets"]
vc = j["카테고리1"].value_counts()
norm = lambda s: re.sub(r"[\s/]+", "", str(s))   # noqa: E731

그룹 = {}
for v, n in vc.items():
    그룹.setdefault(norm(v), []).append((v, n))

print(f"원본 고유값 {len(vc)}개 → 정규화하면 {len(그룹)}개 개념")
print()
print("표기 변종이 있는 카테고리 —")
소수변종 = 0
for k, vs in sorted(그룹.items()):
    if len(vs) > 1:
        vs = sorted(vs, key=lambda t: -t[1])
        소수변종 += sum(n for _, n in vs[1:])
        print("  " + " | ".join(f"{v!r}={n}" for v, n in vs))
print()
print(f"→ 소수 변종에 묶인 행: {소수변종:,}건")
print("  원본 바이트로 조인하면 이만큼이 조용히 빠진다.")

> 🚨 **`data_list.md` 는 `'환경  설비'` 한 쌍만 경고하고 있었다.** 실제로는 **6쌍**이다.
> `'공조/냉각/유공압'`·`'전기/전자/계측'`·`'고무 / 플라스틱'`·`'물류운반/하역'`·`'정보통신/보안'` 도 같다.
>
> ⛔ **그렇다고 원본을 정규화해서 덮어쓰면 안 된다** — `assets.category` 조인 키는
> **원본 바이트**다(D68 · `residual_curve.md §2-6`). **파생 컬럼을 따로 만든다.**

In [ ]:
j = j.copy()
j["카테고리1_정규화"] = j["카테고리1"].map(norm)      # 파생 — 원본은 그대로 둔다
print("원본 유지 확인:", repr(j["카테고리1"].value_counts().index[5]))
print("파생 컬럼    :", repr(j["카테고리1_정규화"].value_counts().index[5]))

### 4-2. 카테고리 매핑 테이블 — 수기 정의

In [ ]:
# 중진공 카테고리(정규화) → 조달청 분야. ⚠ 아래가 이 노트북에서 유일한 "수기" 판단이다.
CATEGORY_MAP = {
    "공작기계": "기계설비",
    "가공기계": "기계설비",
    "공조냉각유공압": "기계설비",
    "환경설비": "기계설비",
    "동력전달기계": "기계설비",
    "금형기계": "기계설비",
    "포장기계": "기계설비",
    "식품관련": "기계설비",
    "물류운반하역": "기계설비",
    "차량운반기계": "기계설비",
    "농기계": "기계설비",
    "고무플라스틱": "기계설비",
    "전기전자계측": "전기정보통신",
    "정보통신보안": "전기정보통신",
    "자동화기기": "전기정보통신",
    # 아래는 의도적으로 매핑하지 않는다 — 어느 쪽인지 판단 근거가 없다
    "기타": None,
    "일반산업": None,
    "소모성자재": None,
}
j["조달청_분야"] = j["카테고리1_정규화"].map(CATEGORY_MAP)

미매핑 = j["조달청_분야"].isna().sum()
print(f"매핑됨   : {j['조달청_분야'].notna().sum():,}건")
print(f"미매핑   : {미매핑:,}건 ({미매핑 / len(j) * 100:.1f}%) — '기타'·'일반산업'·'소모성 자재'")
print()
print(j["조달청_분야"].value_counts(dropna=False).to_string())

### 4-3. ⛔ 그런데 이 매핑으로 join 하면 안 된다

분야는 **2개 값**뿐이다. 16,011건을 3,720건에 분야로 붙이면 **카티전 곱**이 되고
(최대 약 2,900만 행), 그렇게 나온 행은 **아무 의미가 없다** — "공작기계 매물"과
"STS관 단가"가 같은 행에 앉을 뿐이다.

**실제로 필요한 건 품목 수준 매칭인데, 그게 되는지 아래에서 확인한다.**

In [ ]:
print("join 했을 때 나올 행 수 (실행하지 않고 계산만) —")
for 분야, n in j["조달청_분야"].value_counts().items():
    k = (kseis_all["분야"] == 분야).sum()
    print(f"  {분야:12} 중진공 {n:,} × 조달청 {k:,} = {n * k:,}행")
print()
print("⛔ 이건 조인이 아니라 폭발이다. 실행하지 않는다.")

### 4-4. 품목 수준 매칭이 되는가 — MaintQ `parts` 기준 검증

In [ ]:
# 마지막 2개(케이블·전동기)는 **대조군**이다 — 겹치는 게 정말 있는지 확인하기 위해 넣었다.
MAINTQ_PARTS = ["냉각팬", "제어보드", "전원모듈", "히트싱크", "서멀 그리스",
                "흡기 필터", "I/O 확장보드", "모터케이블", "펌프", "베어링",
                "케이블", "전동기"]
print(f"{'MaintQ 부품':<14}{'조달청 품명 매칭':>14}   비고")
print("-" * 70)
매칭 = {}
for p in MAINTQ_PARTS:
    핵심 = p.split()[0]
    hit = kseis_all[kseis_all["품명"].str.contains(핵심, na=False)]
    매칭[p] = int(len(hit))
    예 = sorted(hit["품명"].unique())[:2]
    print(f"{p:<14}{len(hit):>14}   {예 if 예 else '— 없음'}")
print()
영 = [p for p, v in 매칭.items() if v == 0]
print(f"매칭 0건: {len(영)} / {len(매칭)}  → {영}")
print(f"매칭 있음: {[p for p, v in 매칭.items() if v > 0]}")
print()
print("→ 대조군(케이블·전동기)이 잡히므로 검색 자체는 정상 동작한다.")
print("  즉 '검색이 안 된 것'이 아니라 **품목이 실제로 없는 것**이다.")

> 🚨 **`data_list.md` 의 주장이 실측과 어긋난다.**
> 문서는 조달청 xlsx 를 *"냉각팬·케이블·PCB 등 **부품 단가 현실화**"* 로 적어 놨지만,
> **냉각팬 0건 · PCB 0건 · 제어보드 0건 · 전원모듈 0건 · 펌프 0건 · 베어링 0건**이다.
> 실제로 겹치는 건 **케이블 계열**과 **전동기**뿐이다.
>
> 이 데이터는 **"시설공통자재"(건축설비용 배관·케이블·트레이)**이지 **설비 부품 카탈로그가 아니다.**

## 5. 나머지 4종 — 왜 안 합치는가

| 데이터셋 | 다른 데이터와의 관계 | 판정 |
|---|---|---|
| `iros_collateral_stats` | 월별 **집계 통계**. 개별 물건 식별자가 없다 | ⛔ join 불가 — 결합할 입도가 없음 |
| `law_articles` | 법령 조문 8건. `rules.law_refs` 가 참조하지만 그건 **룰 쪽 키**다 | ⛔ 데이터셋 간 join 아님 |
| `dacon_fan_sound` | 팬 소음 wav 메타. 공유 키 0개 | ⛔ 미채택 데이터셋 (`06-1`) |
| `joonggomall_assets` | 조달청과는 §4 대로 품목 매칭 실패 | 🔶 **분야 태그만** 부여 |

In [ ]:
print("각 데이터셋의 '식별자 성격' 컬럼 —")
for k, v in sets.items():
    후보 = [c for c in v.columns if any(t in c for t in
            ["ID", "id", "번호", "ref", "SAMPLE", "slug", "모델", "품명"])]
    print(f"  {k:26} {후보[:4]}")
print()
print("→ 서로 같은 대상을 가리키는 식별자가 한 쌍도 없다.")

## 6. 생성한 파생 컬럼 기록

In [ ]:
generated_columns = [
    {"column": "분야", "logic": "자재구분 문자열에 '기계설비' 포함 여부로 2값 태깅",
     "source_columns": ["자재구분"], "dataset": "kseis_machinery / kseis_electrical"},
    {"column": "단가_표기", "logic": "'{가격:,}원/{단위}' 문자열 — 단위 없는 가격 비교를 막는다",
     "source_columns": ["가격", "단위"], "dataset": "kseis_all"},
    {"column": "가격_로그", "logic": "log10(max(가격,1)) — 가격 분포가 3자릿수 이상 퍼져 있어서",
     "source_columns": ["가격"], "dataset": "kseis_all"},
    {"column": "카테고리1_정규화", "logic": r"re.sub(r'[\s/]+', '', 카테고리1) — 표기 변종 6쌍 흡수. "
                                    "⛔ 원본 카테고리1 은 조인 키라 그대로 둔다(D68)",
     "source_columns": ["카테고리1"], "dataset": "joonggomall_assets"},
    {"column": "조달청_분야", "logic": "CATEGORY_MAP 수기 매핑 (18개념 → 기계설비/전기정보통신/None). "
                                "⚠ 태그일 뿐 join 키로 쓰지 않는다",
     "source_columns": ["카테고리1_정규화"], "dataset": "joonggomall_assets"},
]
for g in generated_columns:
    print(f"  {g['dataset']:32} {g['column']}")

## 7. 저장

In [ ]:
kseis_all.to_parquet(PROCESSED / "kseis_all.parquet", index=False)
j.to_parquet(PROCESSED / "joonggomall_with_category.parquet", index=False)
print(f"✅ kseis_all.parquet                  ({len(kseis_all):,}행 × {len(kseis_all.columns)}열)")
print(f"✅ joonggomall_with_category.parquet  ({len(j):,}행 × {len(j.columns)}열)")

In [ ]:
payload = {
    "joins": [
        {
            "name": "kseis_all",
            "method": "concat (세로 결합)",
            "join_key": None,
            "reason": "같은 파일 포맷의 분야별 배포본 — 컬럼 12개 완전 동일. join 이 아니다",
            "rows_before": {"kseis_machinery": int(len(m)), "kseis_electrical": int(len(e))},
            "rows_after": int(len(kseis_all)),
            "duplicates": int(dup),
        },
        {
            "name": "joonggomall_with_category",
            "method": "map (태그 부여) — join 아님",
            "join_key": "카테고리1_정규화 → CATEGORY_MAP",
            "reason": "분야가 2값뿐이라 실제 join 하면 카티전 곱이 된다. 태그만 붙인다",
            "rows_before": int(len(sets["joonggomall_assets"])),
            "rows_after": int(len(j)),
            "mapped": int(j["조달청_분야"].notna().sum()),
            "unmapped": int(j["조달청_분야"].isna().sum()),
        },
    ],
    "not_joined": {
        "iros_collateral_stats": "월별 집계 통계 — 개별 물건 식별자가 없어 결합할 입도가 없다",
        "law_articles": "법령 조문. rules.law_refs 가 참조하지만 데이터셋 간 join 이 아니다",
        "dacon_fan_sound": "미채택 데이터셋. 공유 키 0개",
    },
    "generated_columns": generated_columns,
    "category_map": CATEGORY_MAP,
    "category_variants_found": 6,
    "category_variant_rows": int(소수변종),
    "maintq_part_match": 매칭,
    "findings": [
        "중진공 카테고리1 에 표기 변종이 6쌍 있다 (문서는 '환경  설비' 1쌍만 경고). "
        f"소수 변종에 {소수변종:,}행이 묶여 있어 원본 바이트 조인 시 조용히 빠진다",
        "조달청 자재는 '시설공통자재'(배관·케이블·트레이)이지 설비 부품 카탈로그가 아니다 — "
        "냉각팬·PCB·제어보드·전원모듈·펌프·베어링 전부 0건",
        "6개 데이터셋을 하나로 묶을 공통 키는 존재하지 않는다",
    ],
}
path = REPORTS / "merge_decisions.json"
path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✅ {path.relative_to(REPO_ROOT)}")